In [1]:
# pip install pyspark

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/local/sdkman/candidates/java/17.0.10-ms"
os.environ["JAVA_TOOL_OPTIONS"] = " ".join([
    "--add-opens=java.base/javax.security.auth=ALL-UNNAMED",
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED",
    "--add-opens=java.base/java.nio=ALL-UNNAMED",
    "--add-opens=java.base/java.lang=ALL-UNNAMED",
    "--add-opens=java.base/java.util=ALL-UNNAMED",
])

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/workspaces/pyspark_udemy_codespace/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL", 
            "jdbc:derby:/workspaces/pyspark_udemy_codespace/metastore_db;create=true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 18:27:18 WARN Utils: Your hostname, codespaces-9ec455, resolves to a loopback address: 127.0.0.1; using 10.0.0.182 instead (on interface eth0)
26/04/17 18:27:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(

Spark version: 4.1.1


In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS spark_db")
spark.sql("USE spark_db")
print("Database created OK")

Database created OK


In [4]:
spark.sql("DROP TABLE IF EXISTS spark_db.diamonds")

import shutil
diamonds_path = '/workspaces/pyspark_udemy_codespace/setup/spark-warehouse/spark_db.db/diamonds'
if os.path.exists(diamonds_path):
    shutil.rmtree(diamonds_path)

spark.sql("""
    CREATE TABLE spark_db.diamonds(
          carat DOUBLE,
          clarity STRING,
          color STRING,
          cut STRING,
          depth STRING,
          price DOUBLE
    )
""")

print("Table created OK")

Table created OK


In [5]:
# Check the first few lines of the file
with open("/workspaces/pyspark_udemy_codespace/data/diamonds.json", "r") as f:
    for i, line in enumerate(f):
        print(line)
        if i > 3:
            break

{"cut":"Ideal","color":"E","clarity":"SI2","carat":0.23,"depth":61.5,"price":326}

{"cut":"Premium","color":"E","clarity":"SI1","carat":0.21,"depth":59.8,"price":326}

{"cut":"Good","color":"E","clarity":"VS1","carat":0.23,"depth":56.9,"price":327}

{"cut":"Premium","color":"I","clarity":"VS2","carat":0.29,"depth":62.4,"price":334}

{"cut":"Good","color":"J","clarity":"SI2","carat":0.31,"depth":63.3,"price":335}



In [6]:
import os

DATA_PATH = os.path.expanduser("/workspaces/pyspark_udemy_codespace/data/diamonds.json")

df = spark.read.json(DATA_PATH)
df.write.mode("overwrite").saveAsTable("spark_db.diamonds")

print("Rows Loaded: ", df.count())
df.show(5)

Rows Loaded:  53940
+-----+-------+-----+-------+-----+-----+
|carat|clarity|color|    cut|depth|price|
+-----+-------+-----+-------+-----+-----+
| 0.23|    SI2|    E|  Ideal| 61.5|  326|
| 0.21|    SI1|    E|Premium| 59.8|  326|
| 0.23|    VS1|    E|   Good| 56.9|  327|
| 0.29|    VS2|    I|Premium| 62.4|  334|
| 0.31|    SI2|    J|   Good| 63.3|  335|
+-----+-------+-----+-------+-----+-----+
only showing top 5 rows


In [7]:
result = spark.sql("""
    select color, avg(price) as avg_price
    from spark_db.diamonds
    group by color
    order by avg_price desc
""")

result.show()

+-----+------------------+
|color|         avg_price|
+-----+------------------+
|    J|  5323.81801994302|
|    I| 5091.874953891553|
|    H| 4486.669195568401|
|    G| 3999.135671271697|
|    F| 3724.886396981765|
|    D|3169.9540959409596|
|    E|3076.7524752475247|
+-----+------------------+



In [8]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/workspaces/pyspark_udemy_codespace/setup/spark-warehouse'

In [9]:
# Check what tables actually exist
spark.sql("SHOW TABLES IN spark_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
| spark_db| diamonds|      false|
+---------+---------+-----------+

